# Federated

In [ ]:
import pandas as pd
import numpy as np
import os
import json
import argparse

In [ ]:
def partition_federated(
    data_path: str,
    output_dir: str,
    n_clients: int = 5,
    seed: int = 42,
):
    print(f"Loading data from {data_path}...")
    df = pd.read_csv(data_path)

    # ── 1. Unique patients ───────────────────────────────────────────────────
    unique_patients = df["patient_nbr"].unique()
    n_patients = len(unique_patients)
    print(f"Total rows       : {len(df):,}")
    print(f"Unique patients  : {n_patients:,}")
    print(f"Unique encounters: {df['encounter_id'].nunique():,}")

    # ── 2. Shuffle patient IDs (reproducible) ────────────────────────────────
    rng = np.random.default_rng(seed)
    shuffled_patients = rng.permutation(unique_patients)

    # ── 3. Split patient IDs into N roughly-equal chunks ────────────────────
    patient_chunks = np.array_split(shuffled_patients, n_clients)

    # ── 4. Write each client's data to its own folder ────────────────────────
    manifest = {}  # for quick sanity checks later

    for i, patient_ids in enumerate(patient_chunks):
        client_dir = os.path.join(output_dir, f"client_{i}")
        os.makedirs(client_dir, exist_ok=True)

        client_df = df[df["patient_nbr"].isin(patient_ids)].copy()
        out_path = os.path.join(client_dir, "data.csv")
        client_df.to_csv(out_path, index=False)

        manifest[f"client_{i}"] = {
            "n_patients" : int(len(patient_ids)),
            "n_encounters": int(len(client_df)),
            "patient_ids_path": out_path,          # data lives here
        }

        print(
            f"  client_{i}: {len(patient_ids):>6,} patients | "
            f"{len(client_df):>7,} encounters → {out_path}"
        )

    # ── 5. Verify zero leakage ────────────────────────────────────────────────
    print("\nVerifying no patient appears in more than one client...")
    all_sets = [set(chunk) for chunk in patient_chunks]
    for a in range(n_clients):
        for b in range(a + 1, n_clients):
            overlap = all_sets[a] & all_sets[b]
            assert len(overlap) == 0, f"LEAK detected between client_{a} and client_{b}!"
    print("✓ Zero leakage confirmed.")

    # ── 6. Save manifest ──────────────────────────────────────────────────────
    manifest_path = os.path.join(output_dir, "manifest.json")
    with open(manifest_path, "w") as f:
        json.dump(manifest, f, indent=2)
    print(f"\nManifest saved to {manifest_path}")

    return manifest